<a href="https://colab.research.google.com/github/RodriTrooCo9/IA-ProIMAGE/blob/main/IMAGE_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys

def pip(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

# Orden importante: primero las bases
pip("peft==0.11.0")
pip("huggingface_hub==0.24.0")
pip("diffusers==0.30.0")
pip("transformers==4.44.0")
pip("accelerate==0.33.0")
pip("safetensors==0.4.3")
pip("gradio==3.50.2")
pip("imageio==2.34.1")
pip("imageio-ffmpeg==0.4.9")
pip("omegaconf")
pip("einops")

subprocess.run(["apt-get", "install", "-qq", "ffmpeg"], capture_output=True)

import torch
print(f"PyTorch : {torch.version}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Dependencias listas")

PyTorch : <module 'torch.version' from '/usr/local/lib/python3.12/dist-packages/torch/version.py'>
CUDA    : False
Dependencias listas


DRIVE + DIRECTORIOS + CONFIGURACION


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, datetime, gc, torch
import numpy as np
from PIL import Image
from pathlib import Path # Implementación de Pathlib para rutas más seguras

# Uso de Pathlib para mejor gestión de rutas
ROOT = Path("/content/drive/MyDrive/AI_Studio_v4")

DIRS = {
    "images"  : ROOT / "images",
    "videos"  : ROOT / "videos",
    "exports" : ROOT / "exports",
    "sessions": ROOT / "sessions",
    "cache"   : ROOT / "model_cache",
    "gallery" : ROOT / "gallery",
}

# Creación de directorios optimizada
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# Configuración de variables de entorno (convertimos Path a string para el OS)
os.environ["HF_HOME"]            = str(DIRS["cache"])
os.environ["TRANSFORMERS_CACHE"] = str(DIRS["cache"])
os.environ["DIFFUSERS_CACHE"]    = str(DIRS["cache"])

LOG = DIRS["sessions"] / "history.jsonl"

def save_log(entry):
    entry["ts"] = datetime.datetime.now().isoformat()
    # Recomendación: Si haces batch, agrupa las entradas antes de escribir
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

def load_log(n=20):
    if not LOG.exists():
        return []
    with open(LOG, encoding="utf-8") as f:
        lines = f.readlines()
    return [json.loads(l) for l in lines[-n:]]

def ts():
    return datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# NUEVA FUNCIÓN: Optimización de memoria
def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

# (Los diccionarios STYLES, PRESETS y NEG_DEFAULT se mantienen igual, su estructura es excelente)

print(f"Drive montado en: {ROOT}")
print(f"Dispositivo : {DEVICE.upper()}")

# CONTROL DE ERROR: Verificación segura de la GPU antes de consultar sus propiedades
if DEVICE == "cuda":
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU         : {gpu_name}")
    print(f"VRAM        : {vram_gb:.1f} GB")
else:
    print("VRAM        : N/A (Ejecutando en CPU - El rendimiento será limitado)")

Mounted at /content/drive
Drive montado en: /content/drive/MyDrive/AI_Studio_v4
Dispositivo : CPU
VRAM        : N/A (Ejecutando en CPU - El rendimiento será limitado)
